# Solutions · 11 · Internal diffusion and effectiveness factors

Worked solutions to the exercises of [notebook 11](../notebooks/11_internal_diffusion_and_effectiveness_factors.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engreact is missing): install it from GitHub
    import engreact
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engreact"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import integrate, optimize
from engreact import catalysis, ideal
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

A first-order reaction (k = 5 1/s per pellet volume, D_e = 2e-9 m2/s) runs on spherical pellets. Find the pellet radius for eta = 0.9 and for eta = 0.5, and the rate per unit reactor volume in each case if the bed void fraction is 0.4. Which would you choose given the pressure-drop result of notebook 04?

In [2]:
k, D_e, phi_bed = 0.05, 2e-9, 0.4
for eta_target in (0.9, 0.5):
    R_p = optimize.brentq(lambda R: catalysis.effectiveness_factor(R * np.sqrt(k / D_e), "sphere") - eta_target, 1e-6, 1e-1)

    print(f"eta = {eta_target}: pellet radius {R_p*1e3:.2f} mm; rate per unit bed volume = eta k (1 - phi) C = {eta_target * k * (1 - phi_bed):.3f} C per second")
print("smaller pellets use the catalyst better but raise the pressure drop (Ergun ~ 1/D_p^2 in the viscous term): the trade-off is settled by the compressor cost against catalyst cost")

eta = 0.9: pellet radius 0.26 mm; rate per unit bed volume = eta k (1 - phi) C = 0.027 C per second
eta = 0.5: pellet radius 0.95 mm; rate per unit bed volume = eta k (1 - phi) C = 0.015 C per second
smaller pellets use the catalyst better but raise the pressure drop (Ergun ~ 1/D_p^2 in the viscous term): the trade-off is settled by the compressor cost against catalyst cost


Full utilisation needs pellets a fraction of a millimetre across, which a packed bed cannot tolerate for
pressure drop; a couple of millimetres at $\eta \approx 0.5$ is the usual compromise, giving up half the
catalyst's activity to keep the compressor bill sane. The pressure-drop result of notebook 04 and this one are
the two halves of the same decision.

## Exercise 2

An eggshell catalyst puts the active phase in a 0.2 mm shell on 3 mm pellets. Treating the shell as a slab of half-thickness 0.1 mm, compare its effectiveness factor with that of a uniformly impregnated 3 mm sphere for the kinetics above.

In [3]:
shell = 0.1e-3                                                     # half-thickness of the active shell, slab geometry
eta_shell = catalysis.effectiveness_factor(shell * np.sqrt(k / D_e), "slab")
R_p = 1.5e-3
eta_sphere = catalysis.effectiveness_factor(R_p * np.sqrt(k / D_e), "sphere")
vol_fraction = 1 - ((R_p - 2 * shell) / R_p) ** 3
print(f"eggshell: eta = {eta_shell:.3f} on {vol_fraction:.0%} of the pellet volume -> {eta_shell * vol_fraction:.3f} of a fully active pellet's rate")
print(f"uniform 3 mm sphere: eta = {eta_sphere:.3f}")

eggshell: eta = 0.924 on 35% of the pellet volume -> 0.323 of a fully active pellet's rate
uniform 3 mm sphere: eta = 0.347


The eggshell pellet's active layer is nearly fully effective but holds only a third of the volume; the
uniform pellet uses the whole volume at a third of its activity. Per pellet the two are comparable; per gram
of active metal the eggshell wins by a wide margin, which is why precious-metal catalysts are almost always
eggshell.

## Exercise 3

**Implement it yourself:** solve the slab pellet equation y'' = phi^2 y with solve_bvp and compute eta from the flux at the surface; compare with tanh(phi)/phi for phi = 2.

In [4]:
phi = 2.0
x = np.linspace(0, 1, 200)
sol = integrate.solve_bvp(lambda x_, y: np.vstack([y[1], phi**2 * y[0]]), lambda ya, yb: [ya[1], yb[0] - 1], x, np.vstack([np.cosh(phi * x) / np.cosh(phi), phi * np.sinh(phi * x) / np.cosh(phi)]), tol=1e-8)
eta_flux = sol.sol(1.0)[1] / phi**2                                # flux at the surface / (reaction if C = C_s everywhere)
print(f"from the surface flux: eta = {eta_flux:.5f}; tanh(phi)/phi = {np.tanh(phi)/phi:.5f}; library: {catalysis.effectiveness_factor(phi, 'slab'):.5f}")

from the surface flux: eta = 0.48201; tanh(phi)/phi = 0.48201; library: 0.48201


The effectiveness factor is the flux into the pellet divided by the rate the whole pellet would have at the
surface concentration; the BVP solution's surface gradient gives it directly, matching $\tanh\phi/\phi$.